In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn
import statsmodels.api as sm
import scipy as sc

In [ ]:
df = pd.read_csv('train.csv')
df = df.set_index('id')
df.timestamp = pd.to_datetime(df.timestamp)
df.lat = df.lat.round(3)
#df.lat = ''.format()
#df = df.fillna(df.mean())

In [ ]:
df.lon = df.lon.round(3)

df.head()

In [ ]:
df_temp = pd.get_dummies(df)
df_temp.head()

In [ ]:
cov = df_temp.corr()
cov_temp = cov[cov!=1].fillna(0)

In [ ]:
temp_min ={w:(cov_temp[w].argmin(),cov_temp[w].min()) for w in cov_temp.index}
temp_max ={w:(cov_temp[w].argmax(),cov_temp[w].max()) for w in cov_temp.index}

In [ ]:
temp_min['price_doc'],temp_max['price_doc']

In [ ]:
df_temp = df.dropna()
df_grouped = df.groupby(['sub_area'])
floor = df_temp.floor
price = df_temp.price_doc

In [ ]:

'''
for name,group in df_grouped:
    plt.figure()
    plt.title(name)
    plt.scatter(group.max_floor,group.price_doc,label=name)

plt.legend()    
plt.show()
'''

In [ ]:
from collections import *
dict_building = defaultdict()
l = lambda x:dict_building[x].append(df.price_doc)

In [ ]:
df_temp['building'] = df_temp.build_year.map(int).map(str)+ df_temp.sub_area + df_temp.material.map(int).map(str)

In [ ]:
df_grouped1 = df_temp.groupby('building')['price_doc'].count()
df_grouped = df_temp.groupby('building')['price_doc'].mean()

In [ ]:
##Mean price grouped by build year, area and material if the count is greater than 10. 

#df_grouped[df_grouped1[df_grouped1>10].index]

In [ ]:
col_train=['floor','full_sq','life_sq','max_floor','num_room','price_doc','area_m','material']
col_maxfloor_train = ['floor','full_sq','life_sq','price_doc','area_m']
df_train = df.loc[:,col_train].dropna()
df_maxfloor = df_train.loc[:,col_maxfloor_train]

In [ ]:
from sklearn.model_selection import train_test_split
X_train_maxfloor, X_test_maxfloor, y_train_maxfloor, y_test_maxfloor = train_test_split(df_maxfloor.values,df_train.max_floor.values)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train_maxfloor)
StandardScaler(copy=True, with_mean=True, with_std=True)
X_train_maxfloor = scaler.transform(X_train_maxfloor)
X_test_maxfloor = scaler.transform(X_test_maxfloor)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes = (30,),solver='sgd',learning_rate_init=0.01,max_iter=500,activation='tanh')

mlp.fit(X_train_maxfloor, y_train_maxfloor)

In [ ]:
float(np.sum(mlp.predict(X_test_maxfloor) == y_test_maxfloor))/float(len(X_test_maxfloor))

In [ ]:
df_maxfloor_pred = df[pd.isnull(df.max_floor)].loc[:,col_maxfloor_train].dropna()

In [ ]:
pred_maxfloor = mlp.predict(X_test_maxfloor)

In [ ]:
cols=['state','price_doc']
state = df[~pd.isnull(df.state)*df.state!=33].loc[:,cols]

In [ ]:
state_grouped = state.groupby('state').describe()

In [ ]:
#state.groupby('state').describe()

In [ ]:
#####Missing Value Functions

In [ ]:
def fill_material(df,method='loc'):
    if method == 'year':
        df_mat = df[~pd.isnull(df.material) * ~pd.isnull(df.build_year)]
        df_mat_grouped = df_mat.groupby(['build_year'])
        mat={}
        for name,group in df_mat_grouped:
            grp = group.groupby('material').count()['price_doc']
            mat[name] = np.argmax(grp)
        df_mat_null = df[pd.isnull(df.material)].set_index('build_year')
        df_mat_null.material = df_mat_null.material.map(mat)
        df_mat_null = df_mat_null.reset_index()
        df_mat_notnull = df[~pd.isnull(df.material)]
        df = pd.concat([df_mat_notnull,df_mat_null])
        #df = df.set_index('id')
    if method =='loc':
        df['latlon'] = [(x,y) for x,y in zip(df.lat,df.lon)]
        df_mat = df[~pd.isnull(df.lat) * ~pd.isnull(df.lon) * ~pd.isnull(df.material)]
        df_mat_grouped = df_mat.groupby(['latlon'])
        mat={}
        for name,group in df_mat_grouped:
            grp = group.groupby('material').count()['price_doc']
            mat[(round(name[0],3),round(name[1],3))] = np.argmax(grp)
        df_mat_null = df[pd.isnull(df.material)]
        df_mat_null.material = df_mat_null.latlon.map(mat).values
        df_mat_notnull = df[~pd.isnull(df.material)]
        df = pd.concat([df_mat_notnull,df_mat_null])
        #df = df.remove('latlon')
    return df


In [ ]:
def fill_life_sq(df):
    df_life_notnull = df[~pd.isnull(df.life_sq) & ~pd.isnull(df.full_sq)]
    df_life_notnull['ratio'] = df_life_notnull.life_sq/df_life_notnull.full_sq
    mean = df_life_notnull.ratio.mean()
    df_life_null = df[pd.isnull(df.life_sq)]
    df_life_null.life_sq = df_life_null.full_sq * mean
    df_life_notnull = df[~pd.isnull(df.life_sq)]
    df = pd.concat([df_life_notnull,df_life_null])
    return df

In [ ]:
def fill_maxfloor(df):
    df['latlon'] = [(x,y) for x,y in zip(df.lat,df.lon)]
    df_maxfloor = df[~pd.isnull(df.lat) & ~pd.isnull(df.lon) & ~pd.isnull(df.max_floor)]
    df_maxfloor_grouped = df_maxfloor.groupby(['latlon'])
    maxf={}
    for name,group in df_maxfloor_grouped:
        grp = group.groupby('max_floor').count()['price_doc']
        maxf[(round(name[0],3),round(name[1],3))] = np.argmax(grp)
    df_maxf_null = df[pd.isnull(df.max_floor)]
    df_maxf_null.max_floor = df_maxf_null.latlon.map(maxf)
    df_maxf_notnull = df[~pd.isnull(df.max_floor)]
    df = pd.concat([df_maxf_notnull,df_maxf_null])
    #df = df.drop('latlon')
    return df

In [ ]:
def fill_buildyear(df,method='loc'):
    if method =='material':
        df_year = df[~pd.isnull(df.material) & ~pd.isnull(df.build_year)]
        df_year_grouped = df_year.groupby('material')
        maxy={}
        for name,group in df_year_grouped:
            grp = group.groupby('build_year').count()['price_doc']
            maxy[name] = np.argmax(grp)
        df_year_null = df[pd.isnull(df.build_year)]
        df_year_null.build_year = df_year_null.material.map(maxy)
        df_year_notnull = df[~pd.isnull(df.build_year)]
        df = pd.concat([df_year_notnull,df_year_null])
        #df = df.drop('latlon')
    elif method =='loc':
        df['latlon'] = [(x,y) for x,y in zip(df.lat,df.lon)]
        df_year = df[~pd.isnull(df.lat) & ~pd.isnull(df.lon) & ~pd.isnull(df.build_year)]
        df_year_grouped = df_year.groupby(['latlon'])
        maxy={}
        for name,group in df_year_grouped:
            grp = group.groupby('build_year').count()['price_doc']
            maxy[(round(name[0],3),round(name[1],3))] = np.argmax(grp)
        df_year_null = df[pd.isnull(df.build_year)]
        df_year_null.build_year = df_year_null.latlon.map(maxy)
        df_year_notnull = df[~pd.isnull(df.build_year)]
        df = pd.concat([df_year_null,df_year_notnull])
    #df = df.drop('latlon')   
    
    return df  


    

In [ ]:
def fill_state(df):
    df_year = df[~pd.isnull(df.state) & ~pd.isnull(df.build_year)]
    df_year_grouped = df_year.groupby('build_year')
    maxy={}
    for name,group in df_year_grouped:
        grp = group.groupby('state').count()['price_doc']
        maxy[name] = np.argmax(grp)
    df_year_null = df[pd.isnull(df.state)]
    df_year_null.build_year = df_year_null.build_year.map(maxy)
    df_year_notnull = df[~pd.isnull(df.state)]
    df = pd.concat([df_year_notnull,df_year_null])
    #df = df.drop('latlon')
    return df

In [ ]:
temp = fill_material(df,'loc')

In [ ]:
temp = fill_buildyear(temp,'loc')

In [ ]:
temp = fill_maxfloor(temp)

In [ ]:
df_macro = pd.read_csv('macro 2.csv')
df_macro.timestamp = pd.to_datetime(df_macro['timestamp'])

In [ ]:
df_macro.head()

In [ ]:
df_temp = pd.merge(df,df_macro,left_on='timestamp',right_on='timestamp')

In [ ]:
df_macro.timestamp = pd.to_datetime(df_macro['timestamp'])

In [ ]:
def add_yearmonth(df_macro):
    df_macro['year'] = df_macro['timestamp'].map(lambda x: x.year)
    df_macro['month'] = df_macro['timestamp'].map(lambda x: x.month)
    df_macro['ym'] = df_macro['year'].map(str) + df_macro['month'].map(str)
    return df_macro

In [ ]:
df = add_yearmonth(df)

In [ ]:
df_macro = add_yearmonth(df_macro)

In [ ]:
df_macro_month = df_macro.groupby('ym').mean()

In [ ]:
df_macro_month = df_macro_month.iloc[:,:-2]

In [ ]:
df_macro_month = df_macro_month.reset_index()
df_macro_month.head()

In [ ]:
df_data_wmacro = pd.merge(df,df_macro_month,left_on='ym',right_on='ym')

In [ ]:
df_train = fill_material(df_data_wmacro)

In [ ]:
df_train = fill_maxfloor(df_train)

In [ ]:
df_train = fill_buildyear(df_train)

In [ ]:
df_train = fill_life_sq(df_train)

In [ ]:
df_train = fill_state(df_train)

In [ ]:
df_test = pd.read_csv('test.csv',parse_dates=['timestamp'])


In [ ]:
# ylog will be log(1+y), as suggested by https://github.com/dmlc/xgboost/issues/446#issuecomment-135555130
ylog_train_all = np.log1p(df_train['price_doc'].values)
id_test = df_test['id']

df_train.drop(['id', 'price_doc'], axis=1, inplace=True)
df_test.drop(['id'], axis=1, inplace=True)

# Build df_all = (df_train+df_test).join(df_macro)
num_train = len(df_train)
df_all = pd.concat([df_train, df_test])
#df_all = pd.merge_ordered(df_all, df_macro, on='timestamp', how='left')
print(df_all.shape)

# Add month-year
month_year = (df_all.timestamp.dt.month + df_all.timestamp.dt.year * 100)
month_year_cnt_map = month_year.value_counts().to_dict()
df_all['month_year_cnt'] = month_year.map(month_year_cnt_map)

# Add week-year count
week_year = (df_all.timestamp.dt.weekofyear + df_all.timestamp.dt.year * 100)
week_year_cnt_map = week_year.value_counts().to_dict()
df_all['week_year_cnt'] = week_year.map(week_year_cnt_map)

# Add month and day-of-week
df_all['month'] = df_all.timestamp.dt.month
df_all['dow'] = df_all.timestamp.dt.dayofweek

# Other feature engineering
df_all['rel_floor'] = df_all['floor'] / df_all['max_floor'].astype(float)
df_all['rel_kitch_sq'] = df_all['kitch_sq'] / df_all['full_sq'].astype(float)

# Remove timestamp column (may overfit the model in train)
df_all.drop(['timestamp'], axis=1, inplace=True)

In [ ]:
# Deal with categorical values
df_numeric = df_all.select_dtypes(exclude=['object'])
df_obj = df_all.select_dtypes(include=['object']).copy()

for c in df_obj: 
    df_obj[c] = pd.factorize(df_obj[c])[0]

df_values = pd.concat([df_numeric, df_obj], axis=1)

In [ ]:
# Convert to numpy values
X_all = df_values.values
print(X_all.shape)

# Create a validation set, with last 20% of data
num_val = int(num_train * 0.2)

X_train_all = X_all[:num_train]
X_train = X_all[:num_train-num_val]
X_val = X_all[num_train-num_val:num_train]
ylog_train = ylog_train_all[:-num_val]
ylog_val = ylog_train_all[-num_val:]

X_test = X_all[num_train:]

df_columns = df_values.columns

print('X_train_all shape is', X_train_all.shape)
print('X_train shape is', X_train.shape)
print('y_train shape is', ylog_train.shape)
print('X_val shape is', X_val.shape)
print('y_val shape is', ylog_val.shape)
print('X_test shape is', X_test.shape)

In [ ]:
X_train_all[np.isnan(X_train_all)]=-99999
ylog_train_all[np.isnan(ylog_train_all)]=-99999
X_train[np.isnan(X_train)]=-99999
ylog_train[np.isnan(ylog_train)]=-99999
X_val[np.isnan(X_val)]=-99999
ylog_val[np.isnan(ylog_val)]=-99999
X_test[np.isnan(X_test)]=-99999

In [ ]:
dtrain_all = xgb.DMatrix(X_train_all, ylog_train_all,missing=-99999)
dtrain = xgb.DMatrix(X_train, ylog_train,missing=-99999)
dval = xgb.DMatrix(X_val, ylog_val,missing=-99999)
dtest = xgb.DMatrix(X_test,missing=-99999)

In [ ]:
xgb_params = {
    'eta': 0.05,
    'max_depth': 5,
    'subsample': 1.0,
    'colsample_bytree': 0.7,
    'objective': 'reg:linear',
    'eval_metric': 'rmse',
    'silent': 1
}

# Uncomment to tune XGB `num_boost_rounds`
partial_model = xgb.train(xgb_params, dtrain, num_boost_round=1000, evals=[(dval, 'val')],
                       early_stopping_rounds=50, verbose_eval=20)

num_boost_round = partial_model.best_iteration

In [ ]:
model = xgb.train(dict(xgb_params, silent=0), dtrain_all, num_boost_round=num_boost_round)

In [ ]:
ylog_pred = model.predict(dtest)
y_pred = np.exp(ylog_pred) - 1

df_sub = pd.DataFrame({'id': id_test, 'price_doc': y_pred})

In [ ]:
df_sub.to_csv('submission.csv', index=False)